# Interactive Lecture, day 2: Simulating $J/\psi$ with PYTHIA

Yesterday we **measured** a peak in real CMS data: $M_{J/\psi} = 3.0847 \pm 0.0004$ GeV/$c^2$, a width of about 31 MeV/$c^2$ and $\approx 8900$ $J/\psi$ mesons.

Today we do the opposite: we **simulate** $J/\psi$ mesons ourselves, where we know the truth, and compare with the data.

### Why do physicists simulate?

1. **To test the analysis**: if we put in a known answer and get it back, the method works.
2. **To understand the detector**: nature produces a $J/\psi$ with one exact mass, but we measure a wide peak. Where does the width come from?
3. **To count what we did not see**: the detector misses some particles. How many $J/\psi$ were really produced?

### Plan

| Step | What we do |
|---|---|
| 1 | Random numbers: what "Monte Carlo" means |
| 2 | Generate proton-proton collisions with **PYTHIA** |
| 3 | Look at the generated $J/\psi \rightarrow \mu^+\mu^-$ |
| 4 | Add a detector: resolution |
| 5 | Add a detector: acceptance |
| 6 | Compare the simulation with yesterday's data |

## Step 1: Random numbers, or why it is called "Monte Carlo"

A simulation of a collision is a chain of random choices: which quarks collide, how hard, in which direction the products fly, when they decay.
A computer program that makes such random choices is called a **Monte Carlo** program, after the famous casino.

The simplest Monte Carlo program is a die. Let's roll one.

In [1]:
import ROOT
import math
ROOT.gStyle.SetOptFit(1111)
%jsroot on

rnd = ROOT.TRandom3(42) # a random number generator. 42 is the "seed": the same seed always gives the same random numbers

n_rolls = 600
h_die   = ROOT.TH1F("h_die", "Rolling a die;result;how many times", 6, 0.5, 6.5) # 6 bins, centred on 1..6
for i in range(n_rolls):
    result = rnd.Integer(6) + 1 # a random integer 0..5, plus 1
    h_die.Fill(result)

can = ROOT.TCanvas()
h_die.Draw("E1")
can.Draw()

**Question**: we rolled 600 times, so each result should appear 100 times. Why not exactly?

<details>
<summary>Click for answer</summary>

Because it is random. As we saw yesterday, a count of $N$ has an uncertainty of $\sqrt{N}$: the bars fluctuate by about $\pm 10$.
</details>

**Follow-up**: change `n_rolls` to 60 and to 60000. What happens to the bars, and what happens to the error bars *relative* to the bars?

Now something closer to physics: we throw random numbers that follow a **Gaussian** with a mean of 3.0969 (the true $J/\psi$ mass) and a width of 0.031 (yesterday's measured width) and fit them with a Gaussian, exactly like yesterday.

In [2]:
h_gaus = ROOT.TH1F("h_gaus", "Random Gaussian numbers;M [GeV/c^{2}];Events", 200, 2, 4)
for i in range(10000):
    h_gaus.Fill( rnd.Gaus(3.0969, 0.031) ) # a random number from a Gaussian with mean 3.0969 and sigma 0.031

h_gaus.Fit("gaus") # fit with the built-in Gaussian, as yesterday
h_gaus.Draw("E1")
can.Draw()

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      16.6158
NDf                       =           23
Edm                       =  3.22795e-08
NCalls                    =           61
Constant                  =       1262.5   +/-   15.6702     
Mean                      =       3.0968   +/-   0.000316044 
Sigma                     =    0.0315503   +/-   0.000231986  	 (limited)


The fit gives back the numbers we put in (within the error bars). This is our first **closure test**: we know the truth, and the analysis finds it.

Everything today follows this idea, only the "truth" will come from a real collision simulation instead of a Gaussian.

## Step 2: PYTHIA, a real collision simulator

[PYTHIA](https://pythia.org) is the program that ATLAS and CMS use to simulate proton-proton collisions.
It knows the quarks and gluons inside the proton, the rules of the strong interaction (QCD), and the decay tables of all particles.
You give it the beam energy and what you are interested in; it gives you complete collision events, particle by particle.

We tell PYTHIA:
- the LHC energy in 2011: 7 TeV (the data we used yesterday),
- to produce **charmonium** (that is $c\bar{c}$ states: $J/\psi$ and its relatives),
- to decay every $J/\psi$ into $\mu^+\mu^-$ (in nature that happens only 6 % of the time; forcing it saves us a lot of waiting),
- to only simulate collisions that are "hard enough" (this also saves time).

Every particle has a code number, the **PDG ID**: a $J/\psi$ is 443, a $\mu^-$ is 13, a $\mu^+$ is $-13$, a $Z$ boson is 23.

In [3]:
import pythia8mc as pythia8

pythia = pythia8.Pythia("", False) # create the generator (False = do not print the long welcome banner)

settings = [
    "Beams:eCM = 7000",          # proton-proton collisions at 7000 GeV = 7 TeV (LHC in 2011)
    "Charmonium:all = on",       # produce charmonium: J/psi and its relatives
    "PhaseSpace:pTHatMin = 5.",  # only "hard" collisions, this makes the generation much faster
    "443:onMode = off",          # 443 = J/psi: switch off all its decay modes ...
    "443:onIfMatch = 13 -13",    # ... and switch back on only J/psi -> mu- mu+
    "Print:quiet = on",          # do not print pages of settings
    "Next:numberCount = 0",      # do not print a line every 1000 events
]
for setting in settings:
    pythia.readString(setting)

pythia.init()
print("PYTHIA is ready")

PYTHIA is ready


### One collision

Let's generate a **single** collision and print the particles PYTHIA created. Do not try to read all of it: a real collision produces hundreds of particles, most of them soft pions.

In [4]:
found = False
while not found:          # generate collisions until we get one that contains a J/psi -> mu mu (not every collision does)
    pythia.next()         # generate one collision
    event = pythia.event  # the list of all particles in this collision
    for i in range(event.size()):
        if event[i].id() == 443 and abs(event[event[i].daughter1()].id()) == 13:
            found = True

print("Number of particles in this collision:", event.size())
print(" index  PDG ID    name         mother   pT [GeV/c]     eta")
for i in range(event.size()):
    p = event[i]
    if 0 < i < 10 or p.id() == 443 or abs(p.id()) == 13: # print the beams and the first partons, plus every J/psi and muon
        print("%6d  %6d  %-12s %6d   %8.2f  %8.2f" % (i, p.id(), p.name(), p.mother1(), p.pT(), p.eta()))

Number of particles in this collision: 822
 index  PDG ID    name         mother   pT [GeV/c]     eta
     1    2212  p+                0       0.00     54.91
     2    2212  p+                0       0.00    -54.91
     3      21  g                10       0.00     49.05
     4      21  g                11       0.00    -50.30
     5  9940003  J/psi[3S1(8)]      3       9.87      0.60
     6      21  g                 3       9.87     -1.87
     7  9940003  J/psi[3S1(8)]      5      10.47     -0.23
     8      21  g                 5       6.23      0.47
     9      21  g                 6       8.06     -1.87
   380     443  J/psi           295       3.87     -0.32
   647      13  mu-             380       0.79     -0.37
   648     -13  mu+             380       4.15     -0.23


**Questions**
- The first two lines are the two protons of the beams. What is their $p_T$? Why?
- Find the $J/\psi$ (ID 443) and its two muons (13 and $-13$). The "mother" column is the index of the particle it came from, so we can follow the family tree. What was the mother of the $J/\psi$?
- The $J/\psi$ often appears several times: PYTHIA keeps a copy every time something happens to it (e.g. a recoil); the last copy is the one that decays.
- Run the cell again. Is the event the same?

## Step 3: Generate many collisions and collect the $J/\psi \rightarrow \mu^+\mu^-$

Now we run PYTHIA many times. In each collision we look for a $J/\psi$ whose daughters are muons, and we store the two muons. That is all a "generator-level" analysis is.

For each muon we save the same things the CMS data file contained yesterday: $p_T$, $\eta$, $\phi$ and the charge $Q$ (and the full four-vector, for the exact mass).

In [5]:
n_collisions = 5000 # takes about 15-20 seconds

h_true_mass = ROOT.TH1F("h_true_mass", "Generated J/#psi (no detector);M_{#mu#mu} [GeV/c^{2}];Events", 200, 2, 4)
h_true_pt   = ROOT.TH1F("h_true_pt",   "Generated J/#psi;p_{T} [GeV/c];Events", 60, 0, 30)

muon_pairs = [] # here we collect the two muons of every J/psi: a list of (muon1, muon2), each a TLorentzVector

for i in range(n_collisions):
    if not pythia.next(): # generate one collision; on a (rare) failure just skip it
        continue
    event = pythia.event
    for k in range(event.size()):
        p = event[k]
        if p.id() != 443:                            # we only care about J/psi ...
            continue
        d1, d2 = p.daughter1(), p.daughter2()       # ... that decayed into two daughters ...
        if abs(event[d1].id()) != 13:                # ... which are muons (the last copy of the J/psi in the record is the one that decays)
            continue
        muon1 = ROOT.TLorentzVector(event[d1].px(), event[d1].py(), event[d1].pz(), event[d1].e()) # four-vectors, as yesterday
        muon2 = ROOT.TLorentzVector(event[d2].px(), event[d2].py(), event[d2].pz(), event[d2].e())
        muon_pairs.append( (muon1, muon2) )
        h_true_mass.Fill( (muon1 + muon2).M() )  # the invariant mass, as yesterday: ALL THE MAGIC HAPPENS HERE
        h_true_pt.Fill( p.pT() )

print("Generated", n_collisions, "collisions and found", len(muon_pairs), "J/psi -> mu+ mu-")

Generated 5000 collisions and found 1974 J/psi -> mu+ mu-


In [6]:
can2 = ROOT.TCanvas("can2", "can2", 1000, 400)
can2.Divide(2, 1)                       # two plots side by side
can2.cd(1); h_true_mass.Draw("E1")
can2.cd(2); ROOT.gPad.SetLogy(); h_true_pt.Draw("E1")  # logarithmic y axis for the pT
can2.Draw()

**Questions**
- Where is the peak? How wide is it? Compare with yesterday's histogram (2 to 4 GeV/$c^2$, 200 bins, exactly the same axis).
- Zoom in on the peak (drag on the plot). The real $J/\psi$ has a natural width of only 0.093 MeV/$c^2$, 300 times smaller than what we measured yesterday.
- Look at the $p_T$ distribution: are most $J/\psi$ fast or slow?

<details>
<summary>Click for answer</summary>

The generated mass is (almost) exactly 3.0969 GeV/$c^2$ in every event: a single bin. PYTHIA simulates nature, and in nature the $J/\psi$ mass is a fixed number.
The 31 MeV/$c^2$ width we measured yesterday is therefore not a property of the $J/\psi$. **It is a property of the detector.**
</details>

## Step 4: Add a detector, part 1: resolution

No detector is perfect. CMS measures a muon's momentum from the curvature of its track in the magnetic field, and that curvature has a measurement error.
The simplest model of a detector is therefore: the measured $p_T$ is the true $p_T$ plus a random Gaussian error,

$$ p_T^{\text{measured}} = p_T^{\text{true}} \cdot (1 + \delta), \qquad \delta \sim \text{Gaussian}(0, \sigma_{\text{res}}) $$

where $\sigma_{\text{res}}$ is the **momentum resolution** of the detector (e.g. 0.01 means "1 % precision").
The angles $\eta$ and $\phi$ are measured much more precisely, so we leave them alone.

We then calculate the invariant mass with **yesterday's formula**, from the smeared $p_T$, $\eta$, $\phi$, and fit the peak with **yesterday's fit**.

**Your task**: change `resolution` and re-run the cell until the fitted `Sigma` matches yesterday's 0.031. You are calibrating a detector.

In [7]:
def get_invariant_mass(pt1, eta1, phi1, pt2, eta2, phi2): # the same function as yesterday
    return math.sqrt(2*pt1*pt2*(math.cosh(eta1-eta2)-math.cos(phi1-phi2)))

resolution = 0.01 # relative pT resolution of our toy detector: 0.01 = 1 %.  <--- CHANGE ME

def smear(muon): # "measure" a muon with our imperfect detector
    pt_measured = muon.Pt() * (1 + rnd.Gaus(0, resolution)) # true pT times a random factor close to 1
    measured = ROOT.TLorentzVector()
    measured.SetPtEtaPhiM(pt_measured, muon.Eta(), muon.Phi(), 0.10566) # same angles, muon mass 0.10566 GeV/c^2
    return measured

h_reco_mass = ROOT.TH1F("h_reco_mass", "Simulated J/#psi with detector resolution;M_{#mu#mu} [GeV/c^{2}];Events", 200, 2, 4)

for muon1, muon2 in muon_pairs:
    m1 = smear(muon1)
    m2 = smear(muon2)
    mass = get_invariant_mass(m1.Pt(), m1.Eta(), m1.Phi(), m2.Pt(), m2.Eta(), m2.Phi())
    h_reco_mass.Fill(mass)

sim_fit = ROOT.TF1("sim_fit", "gaus", 2.9, 3.3)
sim_fit.SetParNames("A", "Mu", "Sigma")
sim_fit.SetParameters(100, 3.1, 0.03)
h_reco_mass.Fit("sim_fit", "R")
h_reco_mass.Draw("E1")
can.Draw()
print("Simulated width Sigma = %.4f GeV/c^2   (yesterday's data: 0.0314)" % sim_fit.GetParameter("Sigma"))

Simulated width Sigma = 0.0242 GeV/c^2   (yesterday's data: 0.0314)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      54.8406
NDf                       =           22
Edm                       =  1.27811e-10
NCalls                    =           73
A                         =      312.275   +/-   9.01332     
Mu                        =      3.08357   +/-   0.000583494 
Sigma                     =    0.0242287   +/-   0.000426278  	 (limited)


**Questions**
- Which resolution reproduces the data? Is 1 % a good or a bad detector? (Your bathroom scale is about 1 %.)
- The fitted mean is about 10 MeV/$c^2$ **below** 3.0969, even though we put in exactly 3.0969 and our detector has no bias. Why?

<details>
<summary>Click for answer</summary>

About 1.3 to 1.4 % reproduces the data. Real CMS muons are measured to about 1 % in the central part (barrel) and 2 to 3 % at the sides (endcaps).

The shift (about 10 MeV/$c^2$) comes from **our formula**: $M = \sqrt{2p_{T_1}p_{T_2}(\cosh\Delta\eta-\cos\Delta\phi)}$ assumes the muons are massless. They are not (0.106 GeV/$c^2$),
so the formula gives a slightly too small mass. Replace it by `(m1 + m2).M()` and the fitted mean moves to 3.096. So yesterday's difference between the measured 3.0847 and the true 3.0969 is mostly not the detector at all, it is the approximation in the formula.
This is an example of a **systematic** uncertainty: it does not get smaller with more data, and only a simulation can reveal it.
</details>

## Step 5: Add a detector, part 2: acceptance

A detector also does not see everything. CMS can only measure muons that
- do not disappear down the beam pipe: $|\eta| < 2.4$ (this is where the muon chambers end),
- have enough momentum to reach the muon chambers: roughly $p_T > 1$ GeV/$c$ (look at yesterday's table: no muon has a smaller $p_T$).

If **either** muon fails, the whole $J/\psi$ is lost. The fraction that survives is called the **acceptance**.

In [8]:
eta_max = 2.4 # |eta| coverage of the CMS muon system
pt_min  = 1.0 # GeV/c

h_pt_all  = ROOT.TH1F("h_pt_all",  "Acceptance;J/#psi p_{T} [GeV/c];Events", 60, 0, 30)
h_pt_seen = ROOT.TH1F("h_pt_seen", "",                                       60, 0, 30)

n_seen = 0
for muon1, muon2 in muon_pairs:
    jpsi = muon1 + muon2
    h_pt_all.Fill(jpsi.Pt())
    if abs(muon1.Eta()) < eta_max and abs(muon2.Eta()) < eta_max and muon1.Pt() > pt_min and muon2.Pt() > pt_min:
        n_seen += 1
        h_pt_seen.Fill(jpsi.Pt())

acceptance = n_seen / len(muon_pairs)
print("Seen %d of %d J/psi: acceptance = %.1f %%" % (n_seen, len(muon_pairs), 100*acceptance))

h_pt_seen.SetLineColor(ROOT.kRed)
h_pt_all.Draw("E1")
h_pt_seen.Draw("E1 SAME")
legend = ROOT.TLegend(0.55, 0.7, 0.88, 0.88)
legend.AddEntry(h_pt_all,  "all generated J/#psi", "l")
legend.AddEntry(h_pt_seen, "both muons in CMS",    "l")
legend.Draw()
can.Draw()

Seen 678 of 1974 J/psi: acceptance = 34.3 %


**Questions**
- Yesterday we counted $8898 \pm 168$ $J/\psi$ in the data. Using the acceptance, how many were produced in those collisions?
- Which $J/\psi$ do we lose most: slow or fast ones? Why? (Hint: think about where the muons of a slow $J/\psi$ go.)
- Change `pt_min` to 3 GeV/$c$. How much does the acceptance change? This is why experiments spend years measuring their own efficiency.

<details>
<summary>Click for answer</summary>

$N_{\text{produced}} = N_{\text{measured}} / \text{acceptance}$. This is how every "cross-section" measurement at the LHC works: you count what you see and divide by what the simulation says you *could* see.
</details>

## Step 6: Simulation versus data

Let's put everything together: **generation** (PYTHIA) + **resolution** (Step 4) + **acceptance** (Step 5), and draw the result on top of yesterday's real data, using the same histogram and the same formula.
The simulation is scaled so that both peaks have the same height.

In [9]:
import os
csv_file = "data/Jpsimumu.csv" # the same CMS Open Data file as yesterday
if not os.path.exists(csv_file):
    csv_file = "https://opendata.cern.ch/record/5203/files/Jpsimumu.csv"

data = ROOT.RDF.FromCSV(csv_file)
data.Snapshot("my_tree", "my_data.root")
tree = ROOT.TChain("my_tree")
tree.Add("my_data.root")

h_data = ROOT.TH1F("h_data", "Data (points) vs simulation (red);M_{#mu#mu} [GeV/c^{2}];Events", 200, 2, 4)
for i in range(tree.GetEntries()):
    tree.GetEntry(i)
    h_data.Fill( get_invariant_mass(tree.pt1, tree.eta1, tree.phi1, tree.pt2, tree.eta2, tree.phi2) )

h_sim = ROOT.TH1F("h_sim", "", 200, 2, 4)
for muon1, muon2 in muon_pairs:
    m1 = smear(muon1)
    m2 = smear(muon2)
    if abs(m1.Eta()) < eta_max and abs(m2.Eta()) < eta_max and m1.Pt() > pt_min and m2.Pt() > pt_min: # acceptance
        h_sim.Fill( get_invariant_mass(m1.Pt(), m1.Eta(), m1.Phi(), m2.Pt(), m2.Eta(), m2.Phi()) )

h_sim.Scale(h_data.GetMaximum() / h_sim.GetMaximum()) # same peak height as the data
h_sim.SetLineColor(ROOT.kRed)
h_sim.SetFillColorAlpha(ROOT.kRed, 0.3)

h_data.Draw("E1")
h_sim.Draw("HIST SAME")
can.Draw()

**Question**: what is in the data but not in the simulation?

<details>
<summary>Click for answer</summary>

Two things:
1. **Background.** The simulation contains only $J/\psi$. In the data there are also muon pairs that do not come from a $J/\psi$ at all (for example two muons from two different decays in the same collision). They form the flat "floor" we fitted with a straight line yesterday.
2. **The $\psi(2S)$** at 3.69 GeV/$c^2$, which we did not ask PYTHIA to decay into muons (its PDG ID is 100443, try it!).

A real analysis simulates all of these separately and adds them up. If the sum matches the data, we understand our detector and our physics; if not, we have found something new (or a bug).
</details>

# That is how a particle physics measurement is checked

## Take-home messages
1. A **Monte Carlo** simulation makes random collisions where we know the truth, so we can test whether our analysis finds it.
2. The width of a peak is (almost always) the **detector resolution**, not the particle.
3. A detector never sees everything: to count particles we divide by the **acceptance**, which comes from simulation.
4. The chain **generator → detector simulation → same analysis as the data** is exactly what CMS does, only their detector simulation is a few million lines of code instead of two.

# Additional task

Simulate the $Z$ boson with PYTHIA and compare with yesterday's additional task (the fit to https://opendata.cern.ch/record/5208).

Hints: the process is `WeakSingleBoson:ffbar2gmZ = on`, the $Z$ has PDG ID 23, and you should restrict the mass range with `PhaseSpace:mHatMin = 60.` and `PhaseSpace:mHatMax = 120.`.

The interesting question: the $Z$ boson has a natural width of 2.5 GeV/$c^2$. Is it visible in the simulation, which has no detector? Does the detector resolution matter as much as it did for the $J/\psi$?

<details>
<summary>Solution</summary>

```python
pythia_z = pythia8.Pythia("", False)
for setting in ["Beams:eCM = 7000",
                "WeakSingleBoson:ffbar2gmZ = on",   # q qbar -> Z (or a photon)
                "PhaseSpace:mHatMin = 60.",         # only produce masses between 60 ...
                "PhaseSpace:mHatMax = 120.",        # ... and 120 GeV/c^2
                "23:onMode = off",                  # 23 = Z boson: switch off all decays ...
                "23:onIfMatch = 13 -13",            # ... except Z -> mu- mu+
                "Print:quiet = on", "Next:numberCount = 0"]:
    pythia_z.readString(setting)
pythia_z.init()

h_z = ROOT.TH1F("h_z", "Generated Z (no detector);M_{#mu#mu} [GeV/c^{2}];Events", 200, 60, 120)
for i in range(5000):
    if not pythia_z.next():
        continue
    event = pythia_z.event
    for k in range(event.size()):
        p = event[k]
        if p.id() != 23:
            continue
        d1, d2 = p.daughter1(), p.daughter2()
        if abs(event[d1].id()) != 13:
            continue
        muon1 = ROOT.TLorentzVector(event[d1].px(), event[d1].py(), event[d1].pz(), event[d1].e())
        muon2 = ROOT.TLorentzVector(event[d2].px(), event[d2].py(), event[d2].pz(), event[d2].e())
        h_z.Fill( (muon1 + muon2).M() )

z_fit = ROOT.TF1("z_fit", "gaus", 85, 97)
z_fit.SetParameters(500, 91, 3)
h_z.Fit("z_fit", "R")
h_z.Draw("E1")
can.Draw()
```

Even without any detector the $Z$ peak is about 2.5 GeV/$c^2$ wide: this time the width **is** the particle (the $Z$ lives only $3 \cdot 10^{-25}$ s, and a short lifetime means a broad mass, by the uncertainty principle).
A 1.5 % momentum resolution adds only about 1 GeV/$c^2$ to that, so for the $Z$ the detector matters much less than for the $J/\psi$.
</details>

In [10]:
# Your code here